In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

In [2]:
spark = (
    SparkSession.builder
    .appName("desafio_05_particoes")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/19 11:37:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
df = spark.read.csv("/workspace/datasets/05_particoes/clientes.csv", header= True, inferSchema= True)


+----------+----------+-----+----------------+------+-------------+------------+----+-----+-------------+
|cliente_id|      nome|idade|          cidade|estado|    profissao|renda_mensal|sexo|ativo|data_cadastro|
+----------+----------+-----+----------------+------+-------------+------------+----+-----+-------------+
|         1| Cliente 1|   32|Feira de Santana|    BA|   Engenheiro|    12925.67|   M|false|   2025-02-22|
|         2| Cliente 2|   32|Feira de Santana|    BA|   Engenheiro|     4852.23|   M|false|   2024-01-16|
|         3| Cliente 3|   29|Feira de Santana|    BA|   Engenheiro|      8919.6|   F|false|   2023-02-08|
|         4| Cliente 4|   64|        Salvador|    BA|     Analista|     4253.53|   F|false|   2023-01-03|
|         5| Cliente 5|   60|        Salvador|    BA|   Engenheiro|     2710.92|   M|false|   2023-09-15|
|         6| Cliente 6|   30|        Salvador|    BA|   Engenheiro|    10282.08|   F|false|   2023-01-29|
|         7| Cliente 7|   46|Feira de Santana|

In [4]:
df.rdd.getNumPartitions()

1

Pergunta 1

Quantas partições existem inicialmente?

Resposta: Apenas 1 partição 

In [5]:
df_resultado = (
    df.select("cliente_id", "nome", "renda_mensal", "ativo", "cidade", "estado")
    .filter(F.col("ativo") == "true")
    .withColumn("renda_anual", F.col("renda_mensal") * 12)
    .withColumn(
        "cidadania_cliente",
        F.when(F.col("cidade") == "Salvador", "Soteropolitano")
        .when(F.col("cidade") == "Feira de Santana", "Feirense")
        .otherwise("Paulista")
    )
    
)

In [6]:
df_resultado.explain(True)

== Parsed Logical Plan ==
'Project [cliente_id#17, nome#18, renda_mensal#23, ativo#25, cidade#20, estado#21, renda_anual#105, CASE WHEN ('cidade = Salvador) THEN Soteropolitano WHEN ('cidade = Feira de Santana) THEN Feirense ELSE Paulista END AS cidadania_cliente#113]
+- Project [cliente_id#17, nome#18, renda_mensal#23, ativo#25, cidade#20, estado#21, (renda_mensal#23 * cast(12 as double)) AS renda_anual#105]
   +- Filter (ativo#25 = cast(true as boolean))
      +- Project [cliente_id#17, nome#18, renda_mensal#23, ativo#25, cidade#20, estado#21]
         +- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Analyzed Logical Plan ==
cliente_id: int, nome: string, renda_mensal: double, ativo: boolean, cidade: string, estado: string, renda_anual: double, cidadania_cliente: string
Project [cliente_id#17, nome#18, renda_mensal#23, ativo#25, cidade#20, estado#21, renda_anual#105, CASE WHEN (cidade#20 = Salvador

Pergunta 2

Todas as transformações da sua pipeline são Narrow?

Justifique.

Resposta: Sim todas as transformações são Narrow, pois como a pipeline so tem uma partição, as transformações não percorrem mais de uma partição

Pergunta 3

Existe Shuffle?

Explique por quê.

Resposta: Não existe pois não estamos distribuindo dados entre partições

In [7]:
df_resultado_partition = df_resultado.repartition(2)

In [8]:
df_resultado_partition.rdd.getNumPartitions()

2

Pergunta 4

Quantas partições existem agora?

Resposta: Existem 2 partições agora.

In [9]:
df_resultado_partition.explain(True)

== Parsed Logical Plan ==
Repartition 2, true
+- Project [cliente_id#17, nome#18, renda_mensal#23, ativo#25, cidade#20, estado#21, renda_anual#105, CASE WHEN (cidade#20 = Salvador) THEN Soteropolitano WHEN (cidade#20 = Feira de Santana) THEN Feirense ELSE Paulista END AS cidadania_cliente#113]
   +- Project [cliente_id#17, nome#18, renda_mensal#23, ativo#25, cidade#20, estado#21, (renda_mensal#23 * cast(12 as double)) AS renda_anual#105]
      +- Filter (ativo#25 = cast(true as boolean))
         +- Project [cliente_id#17, nome#18, renda_mensal#23, ativo#25, cidade#20, estado#21]
            +- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Analyzed Logical Plan ==
cliente_id: int, nome: string, renda_mensal: double, ativo: boolean, cidade: string, estado: string, renda_anual: double, cidadania_cliente: string
Repartition 2, true
+- Project [cliente_id#17, nome#18, renda_mensal#23, ativo#25, cidade#20

Pergunta 5

O que apareceu de novo no Physical Plan?

Explique o motivo.

Resposta: No Physical Plan apareceram as operações Exchange e RoundRobinPartitioning, indicando que o Spark redistribuiu os registros entre as partições para atender ao repartition(2). O AdaptiveSparkPlan apenas envolve o plano de execução e faz parte do mecanismo de Adaptive Query Execution, não sendo consequência direta da criação de novas partições.

In [10]:
df_resultado_coalesce = df_resultado_partition.coalesce(1)

In [11]:
df_resultado_coalesce.rdd.getNumPartitions()

1

Pergunta 6

O que mudou em relação ao passo anterior?

Resposta: Como a função coalesce tem como objetivo diminuir o numero de partições, agora a pipeline tem apenas 1 partição, reduzindo em 1 o numero de partições

Pergunta 7

Por que essa operação tende a ser mais barata?

Explique utilizando os conceitos de Shuffle e movimentação de dados.

Resposta: O coalesce() tende a ser mais barato porque normalmente reduz a quantidade de partições sem realizar uma redistribuição completa dos dados (Shuffle). Dessa forma, há menos movimentação entre partições, reduzindo o custo de processamento.

In [12]:
df_resultado_coalesce.show()

+----------+-----------+------------+-----+----------------+------+------------------+-----------------+
|cliente_id|       nome|renda_mensal|ativo|          cidade|estado|       renda_anual|cidadania_cliente|
+----------+-----------+------------+-----+----------------+------+------------------+-----------------+
|        67| Cliente 67|     2079.64| true|       São Paulo|    SP|          24955.68|         Paulista|
|       240|Cliente 240|    12758.26| true|        Salvador|    BA|         153099.12|   Soteropolitano|
|        10| Cliente 10|     8888.41| true|Feira de Santana|    BA|         106660.92|         Feirense|
|        50| Cliente 50|     7504.54| true|        Salvador|    BA|          90054.48|   Soteropolitano|
|        55| Cliente 55|    11394.03| true|        Salvador|    BA|136728.36000000002|   Soteropolitano|
|       283|Cliente 283|    14523.47| true|Feira de Santana|    BA|174281.63999999998|         Feirense|
|       234|Cliente 234|     6564.98| true|       São P

Pergunta 8

Em qual momento a DAG realmente foi executada?

Resposta: Ela foi executada apos uma action, nesse caso foi utilizado o show()

Imagine o seguinte cenário.

Você possui:

500 milhões de registros;
200 partições.

No final da pipeline alguém decide gravar apenas um arquivo.

Pergunta 9

Como você faria?

Justifique tecnicamente sua decisão.

Não basta responder qual método usaria. Explique o impacto na execução do Spark.

Resposta: Se eu quisesse escrever apenas um arquivo eu usaria o metodo coalesce, tendo em vista que, seria escrito um arquivo para cada partição, entao como eu so quero 1 arquivo, eu teria que ter apenas 1 partição

Observe sua própria pipeline construída no desafio.

Identifique:

Quais operações são Narrow?

Resposta: A maioria das operações são narrow, visto que elas não fazem alterações entre patições, pois a principio tinhamos apenas uma partição, as operações narrow eram, select(), filter(), withColumn()

Existe alguma Wide Transformation?

Resposta: Existe, quando foi criada outra partição, essa operação é uma wide

Em qual etapa ocorreu o primeiro Shuffle?

Resposta: Ocorreu apos a criação de uma nova partição, (.repartition()) os dados foram distribuidos entre partições, criando o primeiro shuffle

Em que momento a quantidade de partições mudou?

Resposta: Mudou em duas ocasiões, quando utilizei o repartition() para aumentar o numero de partições e quando eu usei o colaesce() para diminuir o numero de partições

Qual foi a Action responsável por executar a DAG?

Resposta: A action resposanvel por executar a DAG foi a função .show()

Explique para um colega da faculdade:

Qual é a diferença entre repartition() e coalesce()?

Não use definições decoradas.

Use uma explicação que algué
m que nunca utilizou Spark conseguiria entender.

Resposta: Imagine um estoque onde existem vários lotes de produtos (partições) e vários funcionários (executores). Quando utilizamos repartition(), o Spark redistribui os lotes entre os funcionários para equilibrar o trabalho. Já o coalesce() reduz a quantidade de lotes, geralmente sem precisar reorganizar completamente todos os produtos, tornando essa operação mais barata.

In [13]:
spark.stop()